# Evaluate the full deep agent with RAGAS

This notebook evaluates the full deep agent from `4_full_agent.ipynb` with four RAGAS metrics:

- **Faithfulness**: whether claims in the answer are supported by the retrieved context.
- **Answer relevance**: whether the answer addresses the user question.
- **Context precision**: whether the retrieved context is useful and ranked appropriately.
- **Context recall**: whether the retrieved context contains the information needed by the reference answer.

The agent calls in this notebook use Anthropic and Tavily credentials. Evaluation uses an OpenAI model and embeddings, so set the corresponding keys in `../.env` before running the live cells.

In [1]:
import os
import sys
from datetime import datetime

from dotenv import load_dotenv

load_dotenv(os.path.join('..', '.env'), override=True)
sys.path.insert(0, os.path.abspath('../src'))

%load_ext autoreload
%autoreload 2

In [2]:
import sys, os
sys.path.append(os.path.abspath("../src"))

In [3]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

from deep_agents_from_scratch.file_tools import ls, read_file, write_file
from deep_agents_from_scratch.prompts import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from deep_agents_from_scratch.research_tools import get_today_str, tavily_search, think_tool
from deep_agents_from_scratch.state import DeepAgentState
from deep_agents_from_scratch.task_tool import _create_task_tool
from deep_agents_from_scratch.todo_tools import read_todos, write_todos

# model = init_chat_model(model='anthropic:claude-sonnet-4-6', temperature=0.0)
model = init_chat_model(model="openai:gpt-4o-mini", temperature=0.0)

max_concurrent_research_units = 3
max_researcher_iterations = 3
sub_agent_tools = [tavily_search, think_tool]
built_in_tools = [ls, read_file, write_file, write_todos, read_todos, think_tool]
research_sub_agent = {
    'name': 'research-agent',
    'description': 'Delegate research to the sub-agent researcher. Only give this researcher one topic at a time.',
    'prompt': RESEARCHER_INSTRUCTIONS.format(date=get_today_str()),
    'tools': ['tavily_search', 'think_tool'],
}
task_tool = _create_task_tool(sub_agent_tools, [research_sub_agent], model, DeepAgentState)
all_tools = sub_agent_tools + built_in_tools + [task_tool]
INSTRUCTIONS = (
    '# TODO MANAGEMENT\n' + TODO_USAGE_INSTRUCTIONS + '\n\n'
    + '=' * 80 + '\n\n# FILE SYSTEM USAGE\n'
    + FILE_USAGE_INSTRUCTIONS + '\n\n'
    + '=' * 80 + '\n\n# SUB-AGENT DELEGATION\n'
    + SUBAGENT_USAGE_INSTRUCTIONS.format(
        max_concurrent_research_units=max_concurrent_research_units,
        max_researcher_iterations=max_researcher_iterations,
        date=datetime.now().strftime('%a %b %-d, %Y'),
    )
)
agent = create_agent(model, all_tools, system_prompt=INSTRUCTIONS, state_schema=DeepAgentState).with_config({'recursion_limit': 40})

## Collect agent traces

RAGAS expects `user_input`, `response`, `retrieved_contexts`, and `reference`. The full deep agent stores research in its virtual `files` state, so those file contents become the retrieved contexts for evaluation.

In [4]:
def message_text(message):
    content = getattr(message, 'content', message.get('content', '') if isinstance(message, dict) else '')
    if isinstance(content, list):
        return ' '.join(part.get('text', str(part)) if isinstance(part, dict) else str(part) for part in content)
    return str(content)


def run_agent_for_evaluation(question):
    result = agent.invoke({'messages': [{'role': 'user', 'content': question}]})
    response = message_text(result['messages'][-1])
    files = result.get('files', {})
    contexts = [content for content in files.values() if content]
    return {
        'user_input': question,
        'response': response,
        'retrieved_contexts': contexts,
        'agent_state': result,
    }

In [5]:
# Keep references focused on facts the agent can verify through Tavily.
evaluation_cases = [
    {
        'user_input': 'What is the Model Context Protocol, and what problem does it solve?',
        'reference': 'Model Context Protocol is an open protocol for connecting AI applications to external data sources, tools, and workflows through a standardized interface.',
    },
    {
        'user_input': 'What is LangGraph used for, and how does it represent an agent workflow?',
        'reference': 'LangGraph is a framework for building stateful, long-running agent workflows as graphs whose nodes perform work and whose edges control execution.',
    },
    {
        'user_input': 'What is retrieval augmented generation and why is retrieval useful?',
        'reference': 'Retrieval augmented generation retrieves relevant external documents and supplies them to a language model so the model can produce answers grounded in current or specialized information.',
    },
]

# Live calls can take several minutes and use model/search credits.
traces = []
for case in evaluation_cases:
    trace = run_agent_for_evaluation(case['user_input'])
    trace['reference'] = case['reference']
    traces.append(trace)

[(trace['user_input'], len(trace['retrieved_contexts'])) for trace in traces]

[('What is the Model Context Protocol, and what problem does it solve?', 5),
 ('What is LangGraph used for, and how does it represent an agent workflow?',
  6),
 ('What is retrieval augmented generation and why is retrieval useful?', 12)]

In [6]:
from datasets import Dataset

evaluation_dataset = Dataset.from_list([
    {
        'user_input': trace['user_input'],
        'response': trace['response'],
        'retrieved_contexts': trace['retrieved_contexts'],
        'reference': trace['reference'],
    }
    for trace in traces
])
evaluation_dataset.to_pandas()[['user_input', 'response', 'retrieved_contexts', 'reference']]

,user_input,response,retrieved_contexts,reference
0,"What is the Model Context Protocol, and what p...",I have completed the tasks related to your req...,[User Request: What is the Model Context Proto...,Model Context Protocol is an open protocol for...
1,"What is LangGraph used for, and how does it re...",I have completed all tasks related to your req...,"[User request: What is LangGraph used for, and...",LangGraph is a framework for building stateful...
2,What is retrieval augmented generation and why...,### Retrieval Augmented Generation (RAG)\n\nRe...,[User request: What is retrieval augmented gen...,Retrieval augmented generation retrieves relev...


## Run RAGAS metrics

The evaluator model judges the generated answer and context. RAGAS metric names differ slightly across releases, so the import below supports both `ResponseRelevancy` and the older `AnswerRelevancy` name.

In [7]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness

try:
    from ragas.metrics import ResponseRelevancy

    answer_relevance_metric = ResponseRelevancy()
except ImportError:
    from ragas.metrics import AnswerRelevancy

    answer_relevance_metric = AnswerRelevancy()

evaluator_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
evaluator_embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
metrics = [
    Faithfulness(),
    answer_relevance_metric,
    ContextPrecision(),
    ContextRecall(),
]

ragas_result = evaluate(
    evaluation_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)
ragas_result.to_pandas()

/tmp/ipykernel_47022/2234301477.py:3: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness
/tmp/ipykernel_47022/2234301477.py:3: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness
/tmp/ipykernel_47022/2234301477.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness
/tmp/ipyk

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[11]: TimeoutError()


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,"What is the Model Context Protocol, and what p...",[User Request: What is the Model Context Proto...,I have completed the tasks related to your req...,Model Context Protocol is an open protocol for...,1.0,0.766083,0.679167,1.0
1,"What is LangGraph used for, and how does it re...","[User request: What is LangGraph used for, and...",I have completed all tasks related to your req...,LangGraph is a framework for building stateful...,1.0,0.988056,0.710000,1.0
2,What is retrieval augmented generation and why...,[User request: What is retrieval augmented gen...,### Retrieval Augmented Generation (RAG)\n\nRe...,Retrieval augmented generation retrieves relev...,NaN,0.760463,0.808799,NaN


In [8]:
results_df = ragas_result.to_pandas()
metric_columns = [column for column in results_df.columns if column not in {'user_input', 'response', 'retrieved_contexts', 'reference'}]
summary = results_df[metric_columns].mean(numeric_only=True).sort_values(ascending=False).to_frame('mean_score')
summary['interpretation'] = summary.index.map({
    'faithfulness': 'Claims supported by retrieved context',
    'answer_relevancy': 'Answer addresses the question',
    'answer_relevance': 'Answer addresses the question',
    'context_precision': 'Retrieved context is useful and well ranked',
    'context_recall': 'Retrieved context covers the reference answer',
}).fillna('RAGAS metric')
summary

,mean_score,interpretation
faithfulness,1.000000,Claims supported by retrieved context
context_recall,1.000000,Retrieved context covers the reference answer
answer_relevancy,0.838200,Answer addresses the question
context_precision,0.732655,Retrieved context is useful and well ranked


## Reading the scores

Scores are generally between 0 and 1, where higher is better. Inspect individual rows, not only the mean: a high faithfulness score with low context recall usually means the answer is grounded in the retrieved material but the agent failed to retrieve enough evidence. A low context precision score often indicates noisy search results or that the agent did not read the most relevant saved files.